# Ligand Graph Builder



## Relevant packages

### Pytorch Geometric (PyG)
We use PyG, a library built upon PyTorch to easily write and train Graph Neural Networks (GNNs) to generate our ligand graphs. 

In [ ]:
# Install all libraries
! pip install pytorch-lightning wandb rdkit ogb
# install PyG
! pip install torch_geometric

`Might not need this!`Set a random seed to ensure repeatability of experiments

In [ ]:
import random
import numpy as np
import torch

# Random seeds and reproducibility
torch.manual_seed(0)
torch.cuda.manual_seed(0)
np.random.seed(0)
random.seed(0)

## Graph representation

For our project, we use a molecular graph to represent the chromophores. The **nodes** are the heavy atoms of the chromophore (i.e. hydrogen atoms are not considered as nodes). We extract the chromophore atoms from the resolved three dimensional structure of the fluorescent protein in the Protein Data Bank. 
We then build an RDKit molecule from the coordinates so that we can build a molecular graph using atom and bond features from RDKit. 
...As this with **node positions** given by the raw atom euclidian xyz extracted from the resolved crystal structure. The **edges** are the euclidian distances between atoms.

## Graph representation

For our project, we use a molecular graph to represent the chromophores. The **nodes** are the heavy atoms of the chromophore (i.e. hydrogen atoms are not considered as nodes) and the **edges** are the chemical bonds. 

In order to get both the correct bond orders but also retain the 3D geometry of the chromophore in the protein environment (**node/edge features**), we use a hybrid approach: 
1. We use the PDB chromophore code to obtain the SMILES of the chromophores as a chemical skeleton 
2. We align the SMILES to the atomic coordinates in the resolved three dimensional structure in the PDB using RDKit's `AssignBondOrdersFromTemplate`
3. We build the graph from the hybrid RDKit molecule, allowing us to use atom and bond features from RDKit.

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Draw
import requests

IPythonConsole.ipython_useSVG = True  # < use SVGs instead of PNGs
IPythonConsole.drawOptions.addAtomIndices = True  # adding indices for atoms
IPythonConsole.drawOptions.addBondIndices = False  # not adding indices for bonds
IPythonConsole.molSize = 200, 200

First, we need to extract the  chromophore code and pdb block for the respective fluorescent protein from the PDB.

In [ ]:
import re
from pathlib import Path

CHROMOPHORE_CODES = {
    'NRQ', 'CRQ', 'NRP', 'CH6', 'CRO', '5SQ', '4M9', 'CR2', 'OFM', 'CR8',
    'CFY', 'OIM', 'CH7', 'GYS', 'WCR', 'GYC', 'DYG', 'FAD', 'PIA', 'CCY',
    'BLR', 'CRF', 'NYG', 'CR7', 'FMN', 'B2H', 'SWG', 'CSH', 'BJF'
}

def get_chromophore_code(pdb_path: str | Path) -> str | None: 
    """
    Extract the chromophore residue code from a PDB legacy file. Scans HETATM records and returns the first residue name that matches
    a known chromophore code.
    Args:
        pdb_path: Path to the .pdb file.
    Returns:
        The matched residue code (e.g. 'CRO'), or None if not found.
    """
    pdb_path = Path(pdb_path)
    if not pdb_path.exists():
        raise FileNotFoundError(f"PDB file not found: {pdb_path}")

    with open(pdb_path, 'r') as f:
        for line in f:
            if not line.startswith('HETATM'):
                continue
            # PDB format: columns 18-20 (0-indexed 17:20) = residue name
            residue_name = line[17:20].strip()
            if residue_name in CHROMOPHORE_CODES:
                return residue_name

    return None  # no chromophore found

def get_chromophore_pdb_block(
    pdb_path: str | Path,
    residue_code: str,
    chain_id: str | None = None
) -> str:
    """
    Extract all HETATM lines for a given chromophore residue code, formatted as a PDB block ready for RDKit's MolFromPDBBlock.
    Args:
        pdb_path:     Path to the .pdb file.
        residue_code: The chromophore residue name (e.g. 'CRO').
        chain_id:     Optional chain ID to filter by (e.g. 'A'). If None, defaults to the first chain containing the chromophore.
    Returns:
        A string PDB block containing only the chromophore's HETATM lines.
    Raises:
        ValueError: If the residue code is not found, or if the specified chain does not contain the chromophore.
    """
    pdb_path = Path(pdb_path)
    
    # PDB column positions (0-indexed):
    # [17:20] residue name, [21] chain ID, [22:26] residue sequence number

    # --- Pass 1: collect all (chain, resseq) pairs for this chromophore ---
    occurrences: dict[str, set[str]] = {}  # chain_id → set of residue seq numbers
    with open(pdb_path, 'r') as f:
        for line in f:
            if not line.startswith('HETATM'):
                continue
            if line[17:20].strip() != residue_code:
                continue
            chain  = line[21].strip()
            resseq = line[22:26].strip()
            occurrences.setdefault(chain, set()).add(resseq)

    if not occurrences:
        raise ValueError(
            f"Residue '{residue_code}' not found in {pdb_path.name}."
        )

    # --- Resolve chain ---
    if chain_id is None:
        # Default to the first chain encountered (alphabetical for determinism)
        chain_id = sorted(occurrences.keys())[0]
        if len(occurrences) > 1:
            print(
                f"[INFO] '{residue_code}' found in chains "
                f"{sorted(occurrences.keys())} in {pdb_path.name}. "
                f"Defaulting to chain '{chain_id}'. "
                f"Pass chain_id=... to override."
            )
    elif chain_id not in occurrences:
        raise ValueError(
            f"Residue '{residue_code}' not found in chain '{chain_id}' "
            f"of {pdb_path.name}. "
            f"Available chains: {sorted(occurrences.keys())}."
        )

    # --- Pass 2: extract HETATM lines for the resolved chain ---
    lines = []
    with open(pdb_path, 'r') as f:
        for line in f:
            if not line.startswith('HETATM'):
                continue
            if line[17:20].strip() != residue_code:
                continue
            if line[21].strip() != chain_id:
                continue
            lines.append(line)

    return ''.join(lines) + 'END\n'

We use the chromophore code to get the canonical SMILES of the chromophore from the RCSB Chemical Component Dictionary and build an RDKit molecule object from the SMILES. Then, we parse the raw PDB block to obtain the coordinates to build a mol from these coordinates using RDKit's `Chem.MolFromPDBBlock`, and we align the SMILES to the atomic coordinates in the resolved three dimensional structure in the PDB using RDKit's `AssignBondOrdersFromTemplate`.

In [ ]:
def get_ccd_smiles(residue_code: str) -> str: 
    """Fetch canonical SMILES from the RCSB Chemical Component Dictionary."""
    url = f"https://data.rcsb.org/rest/v1/core/chemcomp/{residue_code}"
    data = requests.get(url).json()
    return data["rcsb_chem_comp_descriptor"]["smiles"]

def chromophore_mol_from_pdb(pdb_block: str, residue_code: str) -> Chem.Mol:
    """
    Build a chemically correct, 3D-positioned chromophore molecule.
    Combines CCD bond orders with real PDB coordinates.
    """
    # 1. Get the chemically correct template from CCD
    smiles = get_ccd_smiles(residue_code)
    template = Chem.MolFromSmiles(smiles)

    # 2. Parse the raw PDB block (coordinates only, bonds unreliable)
    raw_mol = Chem.MolFromPDBBlock(pdb_block, sanitize=False, removeHs=False)

    # 3. Assign correct bond orders from the template onto the 3D structure
    mol = AllChem.AssignBondOrdersFromTemplate(template, raw_mol)
    mol = Chem.RemoveHs(mol)  # optional: heavy-atom-only graph
    Chem.SanitizeMol(mol)
    return mol

Add a line of code here that executes all these steps? Or somewhere else?

## Graph construction from the hybrid molecule

We first define the **node and edge features** (atom and bond features)

We use the following **node features** (4-dim) extracted from RDKit:  

| feature | description |
| ---- | ----  |
| atom type  | atomic number |
| degree  | number of directly-bonded neighbor atoms, including H atoms |
| formal charge | integer electronic charge assigned to atom |
| hybridization | sp, sp2, sp3, sp3d, or sp3d2 |

**Edge features** (3-dim) are the following:  

| feature | description |
| ---- | ----  |
| bond type  | single, double, triple, or aromatic |
| stereo | none, any, E/Z or cis/trans |
| conjugated  | whether the bond is conjugated |

In [ ]:
ATOM_FEATURES = {
    'atom_type' : [1, 6, 7, 8, 9, 16],  # elements: H, C, N, O, F
    'degree' : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'formal_charge' : [-5, -4, -3, -2, -1, 0, 1, 2, 3, 4, 5],
    'hybridization' : [
        'SP', 'SP2', 'SP3', 'SP3D', 'SP3D2'
        ],
}

BOND_FEATURES = {
    'bond_type':  ['SINGLE', 'DOUBLE', 'TRIPLE', 'AROMATIC'],
    'stereo':     ['STEREONONE', 'STEREOZ', 'STEREOE', 'STEREOCIS', 'STEREOTRANS', 'STEREOANY'],
    'conjugated': [False, True],
}

def get_atom_fv(atom, pos):
    """
    Converts rdkit atom object to a feature vector of indices + 3D position.
    Args: 
        atom: rdkit atom object
        pos:  np.array of shape (3,) with XYZ coordinates from PDB conformer
   Returns: 
        list of feature values
    """
    atom_fv = [
        ATOM_FEATURES['atom_type'].index(atom.GetAtomicNum()),
        ATOM_FEATURES['degree'].index(atom.GetTotalDegree()),
        ATOM_FEATURES['formal_charge'].index(atom.GetFormalCharge()),
        ATOM_FEATURES['hybridization'].index(str(atom.GetHybridization())),
        float(pos[0]),  # X coordinate (Å)
        float(pos[1]),  # Y coordinate (Å)
        float(pos[2]),  # Z coordinate (Å)
    ]
    return atom_fv

def get_bond_fv(bond):
    """
    Converts rdkit bond object to a feature vector
    Args: 
        bond:  rdkit bond object 
        pos_i: np.array of shape (3,) - position of begin atom
        pos_j: np.array of shape (3,) - position of end atom
    Returns: 
        list of feature values
    """
    dist = float(np.linalg.norm(pos_i - pos_j))
    bond_fv = [
        dist, 
        BOND_FEATURES['bond_type'].index(str(bond.GetBondType())),
        BOND_FEATURES['stereo'].index(str(bond.GetStereo())),
        BOND_FEATURES['conjugated'].index(bond.GetIsConjugated()),
    ]
    return bond_fv

In [ ]:
#DO WE NEED THIS!????
# Show indices of bonds
IPythonConsole.drawOptions.addAtomIndices = False  # not adding indices for atoms
IPythonConsole.drawOptions.addBondIndices = True  # adding indices for bonds
mol

### Molecular graph data

We use [Data](https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.data.Data.html#torch_geometric.data.Data) class in `PyG` to create a graph data for DMF.

In [ ]:
from torch_geometric.data import Data

# convert our data to tensors, which are used for model training
x = torch.tensor(atom_fvs, dtype=torch.float)
edge_index = torch.tensor(edge_index, dtype=torch.long)
edge_attr = torch.tensor(bond_fvs, dtype=torch.float)

chromophore_data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)
chromophore_data

def mol_to_graph(mol: Chem.Mol, residue_name: str, protein_code: str) -> Data:
    """
    Converts a chromophore RDKit molecule to a PyTorch Geometric Data object.

    Args:
        mol:          RDKit molecule with 3D conformer (from PDB)
        residue_name: chromophore residue code (e.g. 'CRO')
        protein_code: PDB ID of the protein (e.g. '1EMA')

    Returns:
        torch_geometric.data.Data graph object
    """
    conf      = mol.GetConformer()
    positions = np.array([conf.GetAtomPosition(i) for i in range(mol.GetNumAtoms())])

    # --- Atom features ---
    atom_fvs = [get_atom_fv(atom, positions[i]) for i, atom in enumerate(mol.GetAtoms())]

    # --- Bond features + edge index ---
    edge_index0, edge_index1 = [], []
    bond_fvs = []
    for bond in mol.GetBonds():
        i, j  = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edge_index0 += [i, j]
        edge_index1 += [j, i]
        fv = get_bond_fv(bond, positions[i], positions[j])
        bond_fvs += [fv, fv]  # one entry per direction

    edge_index = [edge_index0, edge_index1]

    # --- Label ---
    y = residue_name + "_" + protein_code  # e.g. 'CRO_1EMA'

    # --- Convert to tensors ---
    x          = torch.tensor(atom_fvs,   dtype=torch.float)
    edge_index = torch.tensor(edge_index, dtype=torch.long)
    edge_attr  = torch.tensor(bond_fvs,   dtype=torch.float)

    chromophore_data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)
    return chromophore_data